# Joint Environment–Schedule Fused-Wall Decoder v2 — adaptive candidate coverage

v1 fixed the crash, but the PDF output shows the beam still decays to about 0.79 W-bit accuracy after 64 rounds. The problem is local candidate starvation: at 10% orientation noise and about 16 propagate bits, the true orientation often has 1–3 flipped bits relative to the measured orientation, while `topk=8` is too small to cover the likely Hamming ball.

v2 changes the local fused-wall candidate generator to adaptive Hamming-ball coverage:

$$
R=\left\lceil p\epsilon+s\sqrt{p\epsilon(1-\epsilon)}\right\rceil
$$

The purpose is to ensure the true local split is present in the candidate set before schedule/state factors try to select it.

In [ ]:
RUN_RANDOM_SWEEP = True
RUN_COVERAGE_TEST = True
RUN_BEAM_DEMO = True
SEED = 1337
RANDOM_SWEEP_TRIALS = 300
NOISE_LEVELS = [0.00,0.05,0.10,0.15,0.20,0.25,0.30,0.35,0.40]
BEAM_NOISE = 0.10
BEAM_WIDTH = 512
BEAM_MAX_ROUNDS = 64
LOCAL_MAX_CANDIDATES = 1500
LOCAL_RADIUS_SAFETY = 2.25
LAMBDA_WALL_MISMATCH = 4.0
LAMBDA_SCHEDULE = 2.0

In [ ]:
import hashlib, math, random, json, itertools
from dataclasses import dataclass
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
MASK32=0xFFFFFFFF
random.seed(SEED); np.random.seed(SEED)
H0=[0x6a09e667,0xbb67ae85,0x3c6ef372,0xa54ff53a,0x510e527f,0x9b05688c,0x1f83d9ab,0x5be0cd19]
K=[0x428a2f98,0x71374491,0xb5c0fbcf,0xe9b5dba5,0x3956c25b,0x59f111f1,0x923f82a4,0xab1c5ed5,0xd807aa98,0x12835b01,0x243185be,0x550c7dc3,0x72be5d74,0x80deb1fe,0x9bdc06a7,0xc19bf174,0xe49b69c1,0xefbe4786,0x0fc19dc6,0x240ca1cc,0x2de92c6f,0x4a7484aa,0x5cb0a9dc,0x76f988da,0x983e5152,0xa831c66d,0xb00327c8,0xbf597fc7,0xc6e00bf3,0xd5a79147,0x06ca6351,0x14292967,0x27b70a85,0x2e1b2138,0x4d2c6dfc,0x53380d13,0x650a7354,0x766a0abb,0x81c2c92e,0x92722c85,0xa2bfe8a1,0xa81a664b,0xc24b8b70,0xc76c51a3,0xd192e819,0xd6990624,0xf40e3585,0x106aa070,0x19a4c116,0x1e376c08,0x2748774c,0x34b0bcb5,0x391c0cb3,0x4ed8aa4a,0x5b9cca4f,0x682e6ff3,0x748f82ee,0x78a5636f,0x84c87814,0x8cc70208,0x90befffa,0xa4506ceb,0xbef9a3f7,0xc67178f2]
def rotr(x,n): return ((x>>n)|(x<<(32-n)))&MASK32
def shr(x,n): return x>>n
def Ch(x,y,z): return (x&y)^((~x&MASK32)&z)
def Maj(x,y,z): return (x&y)^(x&z)^(y&z)
def Sigma0(x): return rotr(x,2)^rotr(x,13)^rotr(x,22)
def Sigma1(x): return rotr(x,6)^rotr(x,11)^rotr(x,25)
def sigma0(x): return rotr(x,7)^rotr(x,18)^shr(x,3)
def sigma1(x): return rotr(x,17)^rotr(x,19)^shr(x,10)
def hw32(x): return int(x&MASK32).bit_count()
def bits_to_word(bits):
    out=0
    for j,b in enumerate(bits):
        if b: out|=(1<<j)
    return out&MASK32

In [ ]:
def sha256_pad_oneblock(msg: bytes)->bytes:
    ml=len(msg)*8; b=bytearray(msg); b.append(0x80)
    while len(b)%64 != 56: b.append(0)
    b += ml.to_bytes(8,'big')
    if len(b)!=64: raise ValueError(f'{len(msg)} bytes requires {len(b)//64} blocks')
    return bytes(b)
def make_schedule(block: bytes):
    W=[int.from_bytes(block[4*i:4*i+4],'big') for i in range(16)]
    for t in range(16,64): W.append((sigma1(W[t-2])+W[t-7]+sigma0(W[t-15])+W[t-16])&MASK32)
    return W
def sha256_trace_oneblock(msg: bytes):
    block=sha256_pad_oneblock(msg); W=make_schedule(block)
    a,b,c,d,e,f,g,h=H0; states=[(a,b,c,d,e,f,g,h)]; T1s=[]; T2s=[]
    for r in range(64):
        T1=(h+Sigma1(e)+Ch(e,f,g)+K[r]+W[r])&MASK32
        T2=(Sigma0(a)+Maj(a,b,c))&MASK32
        h,g,f,e,d,c,b,a = g,f,e,(d+T1)&MASK32,c,b,a,(T1+T2)&MASK32
        T1s.append(T1); T2s.append(T2); states.append((a,b,c,d,e,f,g,h))
    internal64=states[-1]
    digest_words=[(internal64[i]+H0[i])&MASK32 for i in range(8)]
    digest=b''.join(w.to_bytes(4,'big') for w in digest_words).hex()
    return dict(msg=msg,block=block,W=W,states=states,T1=T1s,T2=T2s,internal64=internal64,digest=digest)
def restore_internal_from_digest_hex(digest_hex):
    words=[int(digest_hex[i:i+8],16) for i in range(0,64,8)]
    return tuple((words[i]-H0[i])&MASK32 for i in range(8))
def local_reverse_closure_from_next(x_next,r):
    a1,b1,c1,d1,e1,f1,g1,h1=x_next
    a,b,c,e,f,g=b1,c1,d1,f1,g1,h1
    T2=(Sigma0(a)+Maj(a,b,c))&MASK32; T1=(a1-T2)&MASK32; d=(e1-T1)&MASK32
    F=(T1-Sigma1(e)-Ch(e,f,g)-K[r])&MASK32
    return dict(a=a,b=b,c=c,d=d,e=e,f=f,g=g,T1=T1,T2=T2,F=F)
def reverse_step_with_W(x_next,W_r,r):
    p=local_reverse_closure_from_next(x_next,r); h=(p['F']-W_r)&MASK32
    return (p['a'],p['b'],p['c'],p['d'],p['e'],p['f'],p['g'],h)
DOS_HELLO=bytes.fromhex('b409ba0c01cd21b44ccd21'+'48656c6c6f2124'+'90'*37)
trace=sha256_trace_oneblock(DOS_HELLO); M=[int.from_bytes(trace['block'][4*i:4*i+4],'big') for i in range(16)]
print('len',len(DOS_HELLO),'digest',trace['digest']); print('hashlib_match',hashlib.sha256(DOS_HELLO).hexdigest()==trace['digest'])
print('M13,M14,M15',[f'0x{x:08x}' for x in M[13:16]])
ok=True
for r in range(64):
    p=local_reverse_closure_from_next(trace['states'][r+1],r)
    ok &= p['F'] == ((trace['states'][r][7]+trace['W'][r])&MASK32)
    ok &= reverse_step_with_W(trace['states'][r+1], trace['W'][r], r) == trace['states'][r]
print('reverse closure validates',ok)

In [ ]:
def adder_trace_operands(h:int,W:int):
    h&=MASK32; W&=MASK32; F=(h+W)&MASK32; c=[0]*33; O={}; cls=[]
    for j in range(32):
        hj=(h>>j)&1; wj=(W>>j)&1; total=hj+wj+c[j]; c[j+1]=1 if total>=2 else 0
        if hj==0 and wj==0: cls.append('K')
        elif hj==1 and wj==1: cls.append('G')
        else: cls.append('P'); O[j]=0 if (hj,wj)==(1,0) else 1
        assert ((F>>j)&1)==(total&1)
    return F,c,O,cls
def required_propagate_bits(F:int,c:List[int]): return [j for j in range(32) if (((F>>j)&1)^c[j])==1]
def reconstruct_from_F_C_O(F:int,c:List[int],O:Dict[int,int]):
    hb=[0]*32; wb=[0]*32
    for j in range(32):
        p=((F>>j)&1)^c[j]
        if p==0: hb[j]=wb[j]=c[j+1]
        else:
            o=O.get(j,0)
            if o==0: hb[j]=1; wb[j]=0
            else: hb[j]=0; wb[j]=1
    return bits_to_word(hb),bits_to_word(wb)
@dataclass
class Leak:
    F_true:int; carries:List[int]; measured_orientation:Dict[int,int]; reliability:float; true_orientation:Dict[int,int]; noise:float
@dataclass(order=True)
class Candidate:
    cost:float; h:int; W:int; orientations:dict; carries:list; prop_count:int; wall_mismatch_hw:int; missing_orientation_bits:int; flip_count:int
def make_leak_for_round(h:int,W:int,noise:float,rng=random):
    F,c,O,cls=adder_trace_operands(h,W); measured={}
    for j,o in O.items(): measured[j]=(1-o) if rng.random()<noise else o
    return Leak(F,c,measured,max(0.51,1-noise),O,noise)
ok=True; props=[]
for _ in range(1000):
    h=random.getrandbits(32); W=random.getrandbits(32); F,c,O,cls=adder_trace_operands(h,W); hh,WW=reconstruct_from_F_C_O(F,c,O)
    ok &= (hh==h and WW==W); props.append(len(O))
print('Exact F+C+O theorem',ok,'mean p',np.mean(props),'min/max',min(props),max(props))

In [ ]:
def adaptive_radius(p:int,noise:float,safety:float=LOCAL_RADIUS_SAFETY):
    if noise<=0: return 0
    mu=p*noise; sd=math.sqrt(max(p*noise*(1-noise),1e-12))
    return min(p,int(math.ceil(mu+safety*sd)))
def adaptive_orientations_from_leak(F_node:int,leak:Leak,max_candidates:int=LOCAL_MAX_CANDIDATES,lambda_wall:float=LAMBDA_WALL_MISMATCH,radius_safety:float=LOCAL_RADIUS_SAFETY):
    required=required_propagate_bits(F_node,leak.carries); pbits=len(required)
    wall_mismatch_hw=hw32(F_node^leak.F_true); wall_penalty=lambda_wall*wall_mismatch_hw
    rel=max(min(leak.reliability,1-1e-12),1e-12); q=1-rel
    base={}; base_cost=wall_penalty; flip_items=[]; missing=0
    for j in required:
        if j in leak.measured_orientation:
            meas=leak.measured_orientation[j]
            c0=-math.log(rel if meas==0 else q); c1=-math.log(q if meas==0 else rel)
        else:
            missing+=1; c0=-math.log(0.5)+1.0; c1=-math.log(0.5)+1.0
        if c0<=c1: base[j]=0; base_cost+=c0; flip_items.append((c1-c0,j))
        else: base[j]=1; base_cost+=c1; flip_items.append((c0-c1,j))
    flip_items.sort(key=lambda x:x[0]); R=adaptive_radius(pbits,leak.noise,radius_safety)
    raw=[]; indices=list(range(len(flip_items)))
    for rad in range(R+1):
        for combo in itertools.combinations(indices,rad):
            O=dict(base); cost=base_cost
            for idx in combo:
                delta,j=flip_items[idx]; O[j]=1-O[j]; cost+=delta
            raw.append((cost,O,rad))
            if len(raw)>=max_candidates: break
        if len(raw)>=max_candidates: break
    raw.sort(key=lambda x:x[0]); out=[]
    for cost,O,rad in raw[:max_candidates]:
        h,W=reconstruct_from_F_C_O(F_node,leak.carries,O)
        out.append(Candidate(cost,h,W,O,leak.carries,pbits,wall_mismatch_hw,missing,rad))
    return out
def measured_orientation_error_count(leak): return sum(1 for j,o in leak.true_orientation.items() if leak.measured_orientation.get(j)!=o)
def local_candidate_coverage(trials=300, noise_levels=NOISE_LEVELS):
    rows=[]
    for noise in noise_levels:
        hit=0; avg_candidates=[]; avg_radius=[]; avg_errors=[]
        for _ in range(trials):
            h=random.getrandbits(32); W=random.getrandbits(32); leak=make_leak_for_round(h,W,noise); cands=adaptive_orientations_from_leak(leak.F_true,leak)
            hit += any(c.h==h and c.W==W for c in cands); avg_candidates.append(len(cands)); avg_radius.append(adaptive_radius(len(leak.true_orientation),noise)); avg_errors.append(measured_orientation_error_count(leak))
        rows.append(dict(noise=noise,true_pair_in_candidates=hit/trials,avg_candidates=float(np.mean(avg_candidates)),avg_radius=float(np.mean(avg_radius)),avg_true_orientation_errors=float(np.mean(avg_errors))))
    return pd.DataFrame(rows)
if RUN_COVERAGE_TEST:
    coverage_df=local_candidate_coverage(300,NOISE_LEVELS); display(coverage_df)
    ax=coverage_df.plot(x='noise',y=['true_pair_in_candidates'],marker='o',figsize=(7,4)); ax.set_ylim(-.05,1.05); ax.grid(alpha=.3); ax.set_title('Adaptive local candidate coverage'); plt.show()

In [ ]:
def single_wall_noise_sweep(trials=300,noise_levels=NOISE_LEVELS):
    rows=[]
    for noise in noise_levels:
        hc=wc=tot=exact=0
        for _ in range(trials):
            h=random.getrandbits(32); W=random.getrandbits(32); leak=make_leak_for_round(h,W,noise); cand=adaptive_orientations_from_leak(leak.F_true,leak,max_candidates=1)[0]
            hc+=32-hw32(cand.h^h); wc+=32-hw32(cand.W^W); tot+=32; exact+=int(cand.h==h and cand.W==W)
        rows.append(dict(noise=noise,h_bit_acc=hc/tot,W_bit_acc=wc/tot,pair_exact=exact/trials))
    return pd.DataFrame(rows)
if RUN_RANDOM_SWEEP:
    sweep_df=single_wall_noise_sweep(RANDOM_SWEEP_TRIALS); display(sweep_df)
    ax=sweep_df.plot(x='noise',y=['h_bit_acc','W_bit_acc','pair_exact'],marker='o',figsize=(8,5)); ax.axhline(.5,ls='--',lw=1); ax.set_title('Single fused-wall top-1 reconstruction'); ax.grid(alpha=.3); plt.show()

In [ ]:
def schedule_pred(Wd:Dict[int,int],t:int): return (sigma1(Wd[t-2])+Wd[t-7]+sigma0(Wd[t-15])+Wd[t-16])&MASK32
def schedule_factor_ready(Wd:Dict[int,int],t:int): return all(k in Wd for k in (t,t-2,t-7,t-15,t-16))
def schedule_residual_hw(Wd:Dict[int,int],t:int): return hw32(Wd[t]^schedule_pred(Wd,t))
Wtrue={i:trace['W'][i] for i in range(64)}; print('true schedule residual sum',sum(schedule_residual_hw(Wtrue,t) for t in range(16,64)))
@dataclass
class BeamNode:
    state:Tuple[int,...]; W:Dict[int,int]; score:float; scored_factors:frozenset; path:Tuple[Tuple[int,int,float],...]
def factors_newly_closed(Wd,scored): return [t for t in range(16,64) if t not in scored and schedule_factor_ready(Wd,t)]
def make_round_leaks_from_trace(tr,noise): return [make_leak_for_round(tr['states'][r][7],tr['W'][r],noise) for r in range(64)]
def joint_beam_decode(tr,noise=BEAM_NOISE,beam_width=BEAM_WIDTH,max_rounds=BEAM_MAX_ROUNDS,max_local_candidates=LOCAL_MAX_CANDIDATES,lambda_schedule=LAMBDA_SCHEDULE,lambda_wall=LAMBDA_WALL_MISMATCH):
    x64=restore_internal_from_digest_hex(tr['digest']); leaks=make_round_leaks_from_trace(tr,noise); trueW=tr['W']
    beam=[BeamNode(x64,{},0.0,frozenset(),tuple())]; hist=[]
    for step,r in enumerate(list(reversed(range(64)))[:max_rounds],1):
        new_nodes=[]
        for node in beam:
            F_node=local_reverse_closure_from_next(node.state,r)['F']
            cands=adaptive_orientations_from_leak(F_node,leaks[r],max_candidates=max_local_candidates,lambda_wall=lambda_wall)
            for cand in cands:
                prev=reverse_step_with_W(node.state,cand.W,r); Wd=dict(node.W); Wd[r]=cand.W; score=node.score+cand.cost; scored=set(node.scored_factors)
                for t in factors_newly_closed(Wd,node.scored_factors): score+=lambda_schedule*schedule_residual_hw(Wd,t); scored.add(t)
                new_nodes.append(BeamNode(prev,Wd,score,frozenset(scored),node.path+((r,cand.W,cand.cost),)))
        if not new_nodes: raise RuntimeError(f'Beam died at step {step}, round {r}')
        new_nodes.sort(key=lambda n:n.score); beam=new_nodes[:beam_width]; best=beam[0]; keys=sorted(best.W); bit_err=sum(hw32(best.W[k]^trueW[k]) for k in keys); true_in=any(all(n.W[k]==trueW[k] for k in n.W) for n in beam)
        hist.append(dict(step=step,round=r,beam_size=len(beam),best_score=best.score,best_known_word_acc=float(np.mean([best.W[k]==trueW[k] for k in keys])) if keys else 0.0,best_total_hw_error=bit_err,best_W_bit_acc=1.0-bit_err/(32*len(keys)) if keys else 0.0,true_partial_path_in_beam=true_in,closed_schedule_factors_best=len(best.scored_factors)))
    return beam,pd.DataFrame(hist)
if RUN_BEAM_DEMO:
    beam,beam_hist=joint_beam_decode(trace); display(beam_hist.tail(10)); best=beam[0]; keys=sorted(best.W); bit_err=sum(hw32(best.W[k]^trace['W'][k]) for k in keys)
    print('decoded rounds',len(keys)); print('word_acc',np.mean([best.W[k]==trace['W'][k] for k in keys])); print('bit_acc',1.0-bit_err/(32*len(keys))); print('closed_factors',len(best.scored_factors)); print('score',best.score)
    ax=beam_hist.plot(x='step',y=['best_W_bit_acc','true_partial_path_in_beam'],marker='o',figsize=(8,5)); ax.set_ylim(-.05,1.05); ax.set_title('Adaptive joint beam decoder diagnostics'); ax.grid(alpha=.3); plt.show()

In [ ]:
results={'digest':trace['digest'],'message_len':len(DOS_HELLO),'M13_M14_M15':[f'0x{x:08x}' for x in M[13:16]],'v2_change':'adaptive Hamming-ball candidate coverage for local fused-wall orientations'}
if 'coverage_df' in globals():
    p='/mnt/data/joint_fused_wall_v2_candidate_coverage.csv'; coverage_df.to_csv(p,index=False); results['candidate_coverage_csv']=p
if 'sweep_df' in globals():
    p='/mnt/data/joint_fused_wall_v2_noise_sweep.csv'; sweep_df.to_csv(p,index=False); results['noise_sweep_csv']=p
if 'beam_hist' in globals():
    p='/mnt/data/joint_fused_wall_v2_beam_history.csv'; beam_hist.to_csv(p,index=False); results['beam_history_csv']=p
p='/mnt/data/joint_fused_wall_v2_summary.json'
with open(p,'w') as f: json.dump(results,f,indent=2)
print(json.dumps(results,indent=2))